# 6. PR Complexity Component Ablation
Run from the repository's **notebooks** directory. Change `INPUT_PATH` below, then **Run All**.
Requires Python 3.10+ with pandas, numpy, scipy, matplotlib, and jinja2.
If needed, run `%pip install pandas numpy scipy matplotlib jinja2` in a separate cell.

This notebook evaluates the realized (post-review) score from notebook #4:
30% code, 20% files, 25% review burden, 10% configuration, and 15% category breadth.
Review burden is 60% review-comment percentile + 40% reviewer percentile.

For each ablation, remove one component and divide the remaining weighted sum by the remaining weight.
**Merge-time rho** is Spearman correlation with merge time.
**Rank agreement** is Spearman correlation with the full score on the same evaluation rows.
These are associations, not evidence of causality or a prediction available at PR creation.
No earlier paper values are hardcoded or treated as expected results.


In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import scipy
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
from IPython.display import display

INPUT_PATH = Path("../data/prs_with_complexity_gpu.csv")
OUTPUT_DIR = Path("../results/ablation")
# "auto": reuse all saved component percentiles if present; otherwise rebuild all.
# "saved": require saved percentiles. "raw": always rebuild from raw fields.
COMPONENT_MODE = "auto"
# None = all rows. 100 = deterministic ~1% sample by integer PR id % 100 == 0.
# Do not apply this again to an already sampled dataset unless intentional.
SAMPLE_MODULUS = None
CHUNK_SIZE = 500_000
# Missing entire raw features cause an error unless explicitly allowed here.
# If True, notebook #4's zero-value fallback is used (constant percentile, NOT zero).
ALLOW_MISSING_FEATURES = False
# Notebook #4 rounds scores to 2 decimals. None selects unrounded scores.
SCORE_DECIMALS = 2
# False retains valid zero-minute merges. Invalid/negative targets are always excluded.
EXCLUDE_ZERO_MERGE_TIME = False

WEIGHTS = {"code": .30, "files": .20, "review": .25, "configuration": .10, "categories": .15}
SAVED_COMPONENTS = {
    "code": "complexity_code_size",
    "files": "complexity_file_breadth",
    "review": "complexity_review_burden",
    "configuration": "complexity_config",
    "categories": "complexity_category_breadth",
}
RAW_COLUMNS = [
    "code_lines_modified", "lines_modified", "files_touched",
    "files_added", "files_deleted", "files_modified",
    "num_review_comments", "num_reviewers", "config_lines_modified",
    "keyword_category_count",
]


## Load only the columns needed
Sampling happens before raw percentile calculation. Saved percentiles retain the reference cohort from notebook #4.
Raw percentiles instead use every loaded row, including rows later excluded for invalid merge time.
Do not mix saved and rebuilt components. Missing raw values become zero and negatives are clipped, matching #4;
infinite raw values are rejected. Missing/negative merge times are excluded rather than converted to zero.


In [ ]:
if COMPONENT_MODE not in {"auto", "saved", "raw"}:
    raise ValueError("COMPONENT_MODE must be auto, saved, or raw.")
if SAMPLE_MODULUS is not None and (
    isinstance(SAMPLE_MODULUS, bool) or not isinstance(SAMPLE_MODULUS, int) or SAMPLE_MODULUS < 1
):
    raise ValueError("SAMPLE_MODULUS must be None or a positive integer.")
header = pd.read_csv(INPUT_PATH, nrows=0).columns.tolist()
if "minutes_to_merge" not in header:
    raise ValueError("Input must contain minutes_to_merge; no target unit is inferred.")
has_saved = all(c in header for c in SAVED_COMPONENTS.values())
mode = ("saved" if has_saved else "raw") if COMPONENT_MODE == "auto" else COMPONENT_MODE
if mode == "saved" and not has_saved:
    raise ValueError("Saved mode needs all five complexity component columns.")
if SAMPLE_MODULUS is not None and "pull_request_id" not in header:
    raise ValueError("Deterministic sampling requires pull_request_id.")

wanted = ["minutes_to_merge", "pr_complexity_score"]
wanted += list(SAVED_COMPONENTS.values()) if mode == "saved" else RAW_COLUMNS
if SAMPLE_MODULUS is not None:
    wanted += ["pull_request_id"]
usecols = [c for c in dict.fromkeys(wanted) if c in header]
parts, source_rows, invalid_sample_ids = [], 0, 0
for chunk in pd.read_csv(
    INPUT_PATH, usecols=usecols, chunksize=CHUNK_SIZE,
    dtype={"pull_request_id": "string"} if SAMPLE_MODULUS is not None else None,
):
    source_rows += len(chunk)
    if SAMPLE_MODULUS is not None:
        # Exact integer parsing avoids float precision loss for large identifiers.
        ids = chunk["pull_request_id"].astype("string").str.strip()
        valid_ids = ids.str.fullmatch(r"[0-9]+", na=False)
        invalid_sample_ids += int((~valid_ids).sum())
        keep = ids.map(
            lambda s: int(s) % SAMPLE_MODULUS == 0 if pd.notna(s) and s.isdigit() else False
        )
        chunk = chunk.loc[keep]
    parts.append(chunk)
df = pd.concat(parts, ignore_index=True)
if len(df) < 3:
    raise ValueError("Need at least three selected PRs.")
print(f"Source rows: {source_rows:,}; selected rows: {len(df):,}; component mode: {mode}")
if invalid_sample_ids:
    warnings.warn(f"Excluded {invalid_sample_ids:,} rows with invalid sampling IDs.")


In [ ]:
quality, missing_features = [], []

def numeric_feature(frame, name, fallback=None):
    source = name if name in frame else fallback if fallback in frame else None
    if source is None:
        if not ALLOW_MISSING_FEATURES:
            raise ValueError(
                f"Missing feature {name!r}. Use the categorized/scored dataset, "
                "or explicitly set ALLOW_MISSING_FEATURES=True for a limited reconstruction."
            )
        missing_features.append(name)
        warnings.warn(f"{name} absent: using a constant zero raw value.")
        values = pd.Series(0.0, index=frame.index)
    else:
        values = pd.to_numeric(frame[source], errors="coerce")
    if np.isinf(values.to_numpy(dtype=float)).any():
        raise ValueError(f"Infinite values found in {source or name}. Clean these first.")
    quality.append({
        "feature": name, "source": source,
        "zero_imputed_rows": int(values.isna().sum()),
        "negative_clipped_rows": int(values.lt(0).sum()),
    })
    return values.fillna(0).clip(lower=0)

def percentile(values):
    # log1p is monotone: it preserves ranks; it does not change percentile results.
    return np.log1p(values).rank(pct=True, method="average")

def build_components(frame):
    if mode == "saved":
        result = frame[list(SAVED_COMPONENTS.values())].apply(pd.to_numeric, errors="coerce")
        if not np.isfinite(result.to_numpy(dtype=float)).all():
            raise ValueError("Saved components contain missing/nonfinite values. Rebuild in raw mode.")
        if not result.ge(0).all().all() or not result.le(1).all().all():
            raise ValueError("Saved components must be percentiles in [0, 1].")
        return result.rename(columns={v:k for k,v in SAVED_COMPONENTS.items()})
    code_size = numeric_feature(frame, "code_lines_modified", "lines_modified")
    if "files_touched" in frame:
        files = numeric_feature(frame, "files_touched")
    else:
        files = sum(numeric_feature(frame, c) for c in ("files_added", "files_deleted", "files_modified"))
    comments = percentile(numeric_feature(frame, "num_review_comments"))
    reviewers = percentile(numeric_feature(frame, "num_reviewers"))
    return pd.DataFrame({
        "code": percentile(code_size),
        "files": percentile(files),
        "review": .60 * comments + .40 * reviewers,
        "configuration": percentile(numeric_feature(frame, "config_lines_modified")),
        "categories": percentile(numeric_feature(frame, "keyword_category_count")),
    })

components = build_components(df)
constant_components = [c for c in components if components[c].nunique() <= 1]
if constant_components:
    warnings.warn(f"Constant components have no ranking information: {constant_components}")
display(components.describe().T)
if quality:
    display(pd.DataFrame(quality))


## Run the ablations
All variants use the same PR rows and the same component percentiles. Only the omitted component and
remaining weight normalization change. A positive delta means removing that component increased
merge-time correlation; this does not automatically make a better general-purpose complexity score.
Constant scores or targets produce undefined correlations (NaN), not zero.


In [ ]:
def weighted_score(component_frame, omitted=None):
    active = {k:w for k,w in WEIGHTS.items() if k != omitted}
    score = 100 * sum(component_frame[k] * w for k,w in active.items()) / sum(active.values())
    return score if SCORE_DECIMALS is None else score.round(SCORE_DECIMALS)

def rho(x, y):
    if len(x) < 3 or pd.Series(x).nunique() < 2 or pd.Series(y).nunique() < 2:
        return float("nan")
    return float(spearmanr(x, y).statistic)

minutes = pd.to_numeric(df["minutes_to_merge"], errors="coerce")
valid = np.isfinite(minutes) & minutes.ge(0)
if EXCLUDE_ZERO_MERGE_TIME:
    valid &= minutes.gt(0)
if int(valid.sum()) < 3:
    raise ValueError("Fewer than three PRs have valid merge times.")
full = weighted_score(components)
target = minutes.loc[valid] / 60
rows = []
for omitted in [None, *WEIGHTS]:
    score = weighted_score(components, omitted).loc[valid]
    rows.append({
        "variant": "Full" if omitted is None else f"Minus {omitted}",
        "omitted_component": omitted or "",
        "n": int(valid.sum()),
        "merge_time_rho": rho(score, target),
        "rank_agreement": rho(score, full.loc[valid]),
        "remaining_weight": sum(w for k,w in WEIGHTS.items() if k != omitted),
    })
results = pd.DataFrame(rows)
results["delta_rho_vs_full"] = results["merge_time_rho"] - results.loc[0, "merge_time_rho"]
display(results.round(4))
print(f"Excluded invalid target rows: {int((~valid).sum()):,}")
if "pr_complexity_score" in df:
    existing = pd.to_numeric(df["pr_complexity_score"], errors="coerce")
    comparable = np.isfinite(existing)
    if comparable.any():
        error = (existing.loc[comparable] - full.loc[comparable]).abs()
        print(f"Existing full score comparison: max absolute difference = {error.max():.6f}")
        if error.max() > .011:
            warnings.warn("Rebuilt full score differs from the saved score. Check cohort, mode, and rounding.")


## Save aggregate outputs
CSV includes deltas and evaluation counts. LaTeX includes the three paper-table columns.
The manifest records settings, source header fingerprint, file size, and cleaning decisions.
No PR text or row-level data is exported. The source fingerprint covers the header, not the entire file.
Cite the generated table in the surrounding paper text, for example:
“Table \ref{tab:ablation} reports component ablations of the realized complexity score.”

A missing category feature creates a constant percentile near 0.5 under notebook #4's fallback;
it does **not** reproduce a score with observed categories. Saved components also do not prove the
original raw features were available. Check the source notebook/data lineage when interpreting them.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results.to_csv(OUTPUT_DIR / "ablation_results.csv", index=False)
paper_table = results[["variant", "merge_time_rho", "rank_agreement"]].rename(columns={
    "variant": "Variant", "merge_time_rho": r"Merge-time $\rho$", "rank_agreement": "Rank agreement",
})
latex = paper_table.to_latex(
    index=False, escape=False, float_format=lambda x: f"{x:.3f}",
    caption="Component ablation results for the realized PR complexity score.",
    label="tab:ablation", position="t", column_format="lrr",
)
(OUTPUT_DIR / "ablation_table.tex").write_text(latex, encoding="utf-8")
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "input_path": str(INPUT_PATH.resolve()),
    "input_size_bytes": INPUT_PATH.stat().st_size,
    "input_modified_ns": INPUT_PATH.stat().st_mtime_ns,
    "input_header_sha256": hashlib.sha256(json.dumps(header).encode()).hexdigest(),
    "source_rows": source_rows, "selected_rows": len(df), "evaluated_rows": int(valid.sum()),
    "sample_modulus": SAMPLE_MODULUS, "invalid_sampling_ids": invalid_sample_ids,
    "component_mode": mode,
    "reference_cohort": "upstream saved percentiles" if mode == "saved" else "all selected rows before target exclusion",
    "weights": WEIGHTS, "review_internal_weights": {"comments": .6, "reviewers": .4},
    "score_decimals": SCORE_DECIMALS, "exclude_zero_merge_time": EXCLUDE_ZERO_MERGE_TIME,
    "allow_missing_features": ALLOW_MISSING_FEATURES, "missing_features": missing_features,
    "constant_components": constant_components, "raw_cleaning": quality,
    "rank_agreement_cohort": "same valid target rows as merge-time correlation",
    "versions": {"python": platform.python_version(), "pandas": pd.__version__,
                 "numpy": np.__version__, "scipy": scipy.__version__},
}
(OUTPUT_DIR / "ablation_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(latex)
print(f"Saved outputs to {OUTPUT_DIR.resolve()}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(results["variant"], results["merge_time_rho"], color="#247a51")
ax.axvline(results.loc[0, "merge_time_rho"], color="black", linestyle="--", linewidth=1)
ax.invert_yaxis()
ax.set_xlabel("Spearman correlation with merge time")
ax.set_title("PR complexity component ablation")
ax.grid(axis="x", alpha=.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "ablation_merge_time.png", dpi=200)
fig.savefig(OUTPUT_DIR / "ablation_merge_time.pdf")
plt.show()
